# C Setup Test

This notebook checks the first C-to-Python bridge in the project.

The C side contains one function, `add_doubles`, which adds two floating-point numbers. The Python side loads the compiled shared library through `ctypes`, declares the expected input and output types, and calls the C function from Python.


## Why C Appears in This Project

Python remains the main language for the project. It is better suited for data loading, cleaning, plotting, notebooks, SQL orchestration, and reporting.

C is included only where it is natural: small numerical routines that may be called many times. The rest of the project uses the same pattern for normal distribution helpers, Black-Scholes pricing, implied-volatility solving, Greeks, Monte Carlo paths, and hedging loops.


## Shared Library

A C source file must be compiled before Python can call it. The compiled output is a shared library: `.so` on Linux, `.dylib` on macOS, and `.dll` on Windows.

The included build commands are in `c_src/build_instructions.md`.


In [ ]:
from pathlib import Path
import platform
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Operating system: {platform.system()}")


## Compile the C File

The notebook attempts to compile the shared library on macOS or Linux. On Windows, using the terminal instructions in `c_src/build_instructions.md` is usually cleaner because compiler setup depends on the local machine.


In [ ]:
system = platform.system()
compiled_dir = PROJECT_ROOT / "compiled"
compiled_dir.mkdir(exist_ok=True)

source_file = PROJECT_ROOT / "c_src" / "example_add.c"

if system == "Darwin":
    output_file = compiled_dir / "libexample_add.dylib"
    command = ["clang", "-shared", "-fPIC", str(source_file), "-o", str(output_file)]
elif system == "Linux":
    output_file = compiled_dir / "libexample_add.so"
    command = ["gcc", "-shared", "-fPIC", str(source_file), "-o", str(output_file)]
else:
    output_file = compiled_dir / "example_add.dll"
    command = None

if command is None:
    print("Use c_src/build_instructions.md to compile on this operating system.")
else:
    result = subprocess.run(command, capture_output=True, text=True)
    print(" ".join(command))
    print(f"Return code: {result.returncode}")
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    print(f"Compiled library exists: {output_file.exists()}")


## Call C from Python

The wrapper in `src/c_bindings.py` loads the shared library and sets the argument and return types for the C function. This type declaration is the part that prevents Python from guessing incorrectly when values cross the Python-C boundary.


In [ ]:
from src.c_bindings import add_doubles_c, add_doubles_python, validate_example_add

a = 2.5
b = 4.25

c_result = add_doubles_c(a, b)
python_result = add_doubles_python(a, b)

print(f"C result: {c_result}")
print(f"Python result: {python_result}")
print(f"Match: {abs(c_result - python_result) < 1e-12}")


In [ ]:
validate_example_add()


## Notes

This example is intentionally small. The point is not that addition needs C. The point is to prove the compilation and binding workflow before adding more important numerical code later.

Every future C function should be checked against a pure Python version. That keeps the C layer useful without making the project harder to trust.
